# 1. What you'll learn

        - compare PR-AUC and recall at a fixed precision
- fit class weights and SMOTE inside training pipelines
- demonstrate duplicate leakage from pre-split oversampling

        **The one question this notebook answers:** How do weights, resampling, and threshold choice affect rare-event decisions—and why is oversampling before splitting leakage?

# 2. Setup

This lesson keeps imbalanced classification small enough to inspect end to end. Fixed seeds and one shared split make every comparison reproducible.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import average_precision_score,precision_recall_curve,precision_score,recall_score,accuracy_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

Five thousand rows contain a 2% positive class with informative and noisy features. Development, validation, and test splits remain stratified and untouched by resampling.

**Small example:** With 5 positives among 100 cases, predicting all negative is 95% accurate and 0% recall. Catching 3 positives with 2 false alarms gives precision 3/5 and recall 3/5.

In [ ]:
X,y=make_classification(n_samples=5000,n_features=14,n_informative=5,n_redundant=3,weights=[.98,.02],class_sep=1.0,flip_y=.004,random_state=SEED);print("Shape:",X.shape,"class counts:",np.bincount(y).tolist(),"positive share:",round(y.mean(),4))

# 4. Explore

The three views establish the target, the useful structure, and the specific data property that will govern the model's success or failure.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].bar(["negative","positive"],np.bincount(y),color=["#176b66","#d38b45"]);axes[0].set_title("Severe imbalance")
sns.histplot(x=X[:,0],hue=y,element="step",stat="density",common_norm=False,ax=axes[1]);axes[1].set_title("One overlapping feature")
axes[2].bar(["all-negative accuracy","all-negative recall"],[1-y.mean(),0],color=[".6","#d38b45"]);axes[2].set_ylim(0,1);axes[2].set_title("Why accuracy misleads");plt.tight_layout();plt.show()

**Takeaway:** A useless all-negative classifier is about 98% accurate.

**Takeaway:** Individual feature distributions overlap heavily.

**Takeaway:** Rare-event evaluation must reward ranking and explicitly expose recall/precision costs.

# 5. Prepare

We create the evaluation boundary before learning any transformation or model choice. Training-only operations remain inside the fitted workflow so held-out rows cannot influence them.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
X_dev,X_test,y_dev,y_test=train_test_split(X,y,test_size=.2,stratify=y,random_state=SEED);X_train,X_val,y_train,y_val=train_test_split(X_dev,y_dev,test_size=.25,stratify=y_dev,random_state=SEED);print("Train/val/test positives:",y_train.sum(),y_val.sum(),y_test.sum())

# 6. Train

        Training turns the assumptions behind imbalanced classification into an explicit fitted rule. The printed intermediate values expose what was learned rather than treating fit() as magic.

        Conceptually, fitting follows these steps:

        1. Fit scaling only on training data.
2. Use class-weighted loss or synthesize minority points inside the training fold.
3. Choose a threshold on validation for an explicit precision/recall requirement.
4. Lock threshold and report untouched test performance.

In [ ]:
plain=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000,random_state=SEED)).fit(X_train,y_train);weighted=make_pipeline(StandardScaler(),LogisticRegression(class_weight="balanced",max_iter=1000,random_state=SEED)).fit(X_train,y_train);smote=ImbPipeline([("scale",StandardScaler()),("smote",SMOTE(random_state=SEED)),("model",LogisticRegression(max_iter=1000,random_state=SEED))]).fit(X_train,y_train)
vp=weighted.predict_proba(X_val)[:,1];precision,recall,thresholds=precision_recall_curve(y_val,vp);valid=np.where(precision[:-1]>=.5)[0];threshold=thresholds[valid[np.argmax(recall[valid])]] if len(valid) else .5;print("Validation-chosen threshold for precision≥.5:",round(float(threshold),3))

# 7. Evaluate

We compare against a deliberately simple baseline, report a task-appropriate metric, and save three diagnostic views that explain where the score came from.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
probs={n:m.predict_proba(X_test)[:,1] for n,m in {"plain":plain,"class weight":weighted,"SMOTE":smote}.items()};aps={n:average_precision_score(y_test,p) for n,p in probs.items()};chosen=probs["class weight"]>=threshold;base=y_test.mean()
fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].bar(["prevalence"]+list(aps),[base]+list(aps.values()),color=[".6","#176b66","#d38b45","#768f8b"]);axes[0].set_title("Test PR-AUC")
for n,p in probs.items():
 pr,re,_=precision_recall_curve(y_test,p);axes[1].plot(re,pr,label=n)
axes[1].axhline(base,ls="--",color="black");axes[1].set(title="Precision–recall",xlabel="recall",ylabel="precision");axes[1].legend()
axes[2].bar(["precision","recall"],[precision_score(y_test,chosen),recall_score(y_test,chosen)],color=["#176b66","#d38b45"]);axes[2].set_ylim(0,1);axes[2].set_title("Locked validation threshold");plt.tight_layout();plt.savefig("key_figure.png",dpi=160,bbox_inches="tight");plt.show();print("Test AP:",{k:round(v,3) for k,v in aps.items()})

**Takeaway:** The model improves on the simple baseline where its assumptions match the data; the diagnostics show that the aggregate score alone is not enough.

# 8. Break it

We now violate the important condition directly: Oversampling before splitting changes test prevalence and lets synthetic near-duplicates straddle train and test, inflating performance. The resulting number makes the cost of that violation visible.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
sm=SMOTE(random_state=SEED);X_all,y_all=sm.fit_resample(X,y);Xa,Xb,ya,yb=train_test_split(X_all,y_all,test_size=.2,stratify=y_all,random_state=SEED);leaky=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000)).fit(Xa,ya);leaky_ap=average_precision_score(yb,leaky.predict_proba(Xb)[:,1])
print(f"Honest original-test SMOTE AP={aps['SMOTE']:.3f}; invalid pre-split-oversampled AP={leaky_ap:.3f}; invalid test prevalence={yb.mean():.1%}")

**Use this when…** positive events are rare and the decision cost can be expressed through ranking, precision, recall, or expected value.

        **Don't use this when…** you plan to optimize raw accuracy or resample before defining honest folds.

        ## Try this

        1. Require precision .8 instead of .5 and observe recall.
2. Compare random oversampling with SMOTE.
3. Calibrate weighted-model probabilities before thresholding.